# Customer Churn Intelligence — EDA and Modelling

**Project:** Customer Churn Intelligence and Retention Decision-Support Platform
**Dataset:** IBM Telco Customer Churn sample (7,043 × 21), Git blob SHA
`3de7a612d1609f25f21a455bda77948729369002`

> The dataset represents a **fictional** telecommunications company. It is not actual
> observed commercial customer data.

This notebook is a readable walk-through of the analysis. The reproducible source of
truth is `src/`: every function called here is the same one the scripts run, so the
notebook and the report can never drift apart. Nothing is reimplemented below.

Run order for the equivalent scripts:

```bash
make validate   # python -m src.data_validation
make eda        # python -m src.eda
make train      # python -m src.train
```

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config, eda, evaluate, schemas, train
from src.data_validation import validate_dataset

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

print("Project root:", PROJECT_ROOT)
print("Raw dataset :", config.RAW_DATASET_PATH.name)

Project root: /Users/krishnamathurm4pro/Desktop/Academics/SDIAM Term 3/SDAIM FINAL PROJECT
Raw dataset : Telco-Customer-Churn.csv


## 1. Data validation

Training refuses to start on a file that fails this contract. The raw file is never
modified: it is read, checked and reported on.

In [2]:
result = validate_dataset()

print("Validation passed:", result.passed)
print("Rows / columns   :", result.summary["rows"], "/", result.summary["columns"])
print("Git blob SHA     :", result.summary["git_blob_sha"])
print("Expected SHA     :", result.summary["expected_git_blob_sha"])
print("Target counts    :", result.summary["target_distribution"])
print("Checks executed  :", len(result.checks))

blank = {k: v for k, v in result.summary["blank_string_counts"].items() if v}
print("Blank strings    :", blank)

Validation passed: True
Rows / columns   : 7043 / 21
Git blob SHA     : 3de7a612d1609f25f21a455bda77948729369002
Expected SHA     : 3de7a612d1609f25f21a455bda77948729369002
Target counts    : {'No': 5174, 'Yes': 1869}
Checks executed  : 30
Blank strings    : {'TotalCharges': 11}


## 2. Loading the data for analysis

Only the documented type corrections are applied: `TotalCharges` is stripped and coerced
to numeric, and `SeniorCitizen` is kept as a category. **No imputation happens here** —
that belongs inside the pipeline so it can be fitted on the training split alone.

In [3]:
frame = eda.prepare_analysis_frame()
print(frame.shape)
frame.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
frame.dtypes.to_frame("dtype")

,dtype
customerID,str
gender,str
SeniorCitizen,str
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


## 3. Target balance

The class imbalance is the single most important fact for metric selection.

In [5]:
target_table = eda.target_distribution(frame)
target_table

,count,percent
Churn,,
No,5174,73.46
Yes,1869,26.54


A model predicting "no churn" for everyone would score the majority-class share as
accuracy while identifying nobody at risk. Accuracy alone is therefore not an acceptable
selection metric; recall, F1 and ROC-AUC are used instead.

## 4. Missing values

In [6]:
missing = eda.missing_value_summary(frame)
missing[missing["missing_after_coercion"] > 0]

,dtype,missing_after_coercion,blank_strings,missing_percent
column,,,,
TotalCharges,float64,11,0,0.1562


In [7]:
# Every blank TotalCharges belongs to a customer with tenure 0 — the blank is
# structurally meaningful, not random.
blank_rows = frame[frame["TotalCharges"].isna()]
print("Rows with missing TotalCharges:", len(blank_rows))
print("Their tenure values:", sorted(blank_rows["tenure"].unique()))

Rows with missing TotalCharges: 11
Their tenure values: [np.int64(0)]


## 5. Numeric distributions by churn status

In [8]:
tenure_stats = eda.numeric_distribution_by_churn(frame, "tenure", "03_tenure_by_churn.png")
tenure_stats

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,37.570,24.114,0.0,15.0,38.0,61.0,72.0
Yes,1869.0,17.979,19.531,1.0,2.0,10.0,29.0,72.0


In [9]:
monthly_stats = eda.numeric_distribution_by_churn(
    frame, "MonthlyCharges", "04_monthlycharges_by_churn.png"
)
total_stats = eda.numeric_distribution_by_churn(
    frame, "TotalCharges", "05_totalcharges_by_churn.png"
)
monthly_stats

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,61.265,31.093,18.25,25.10,64.425,88.4,118.75
Yes,1869.0,74.441,24.666,18.85,56.15,79.650,94.2,118.35


## 6. Churn rate by categorical predictor

In [10]:
contract = eda.churn_rate_by_category(frame, "Contract", "06_churn_rate_by_contract.png")
contract

,customers,churned,churn_rate_percent
Contract,,,
Month-to-month,3875,1655,42.71
One year,1473,166,11.27
Two year,1695,48,2.83


In [11]:
internet = eda.churn_rate_by_category(
    frame, "InternetService", "07_churn_rate_by_internetservice.png"
)
payment = eda.churn_rate_by_category(
    frame, "PaymentMethod", "08_churn_rate_by_paymentmethod.png"
)
tech = eda.churn_rate_by_category(frame, "TechSupport", "09_churn_rate_by_techsupport.png")
security = eda.churn_rate_by_category(
    frame, "OnlineSecurity", "10_churn_rate_by_onlinesecurity.png"
)
payment

,customers,churned,churn_rate_percent
PaymentMethod,,,
Electronic check,2365,1071,45.29
Mailed check,1612,308,19.11
Bank transfer (automatic),1544,258,16.71
Credit card (automatic),1522,232,15.24


Note the confound: the `No internet service` level appears in every add-on column, so the
add-on comparisons partly restate the internet-service split rather than measuring the
add-on itself. These are associations within the sample, not causal effects.

## 7. Numeric correlations

In [12]:
correlation = eda.correlation_matrix(frame)
correlation

,tenure,MonthlyCharges,TotalCharges,ChurnFlag
tenure,1.0000,0.2479,0.8259,-0.3522
MonthlyCharges,0.2479,1.0000,0.6511,0.1934
TotalCharges,0.8259,0.6511,1.0000,-0.1995
ChurnFlag,-0.3522,0.1934,-0.1995,1.0000


`tenure` and `TotalCharges` are strongly related because total charges accumulate with
time. Tree ensembles tolerate this; the linear model uses scaling and regularisation, and
the relationship is recorded as a known limitation.

## 8. Preprocessing and the leakage-safe split

The split happens **once, before any transformer is fitted**. Every imputer, encoder and
scaler lives inside the `Pipeline`, so cross-validation refits them per fold and no test
information can leak into training.

In [13]:
from sklearn.model_selection import train_test_split

model_frame = train.load_model_frame()
X, y = train.split_features_target(model_frame)

print("Feature matrix :", X.shape)
print("customerID in X:", config.ID_COLUMN in X.columns)
print("Churn in X     :", config.TARGET_COLUMN in X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=y,
)
print(f"Train {X_train.shape[0]:,} rows (churn {y_train.mean():.4f})")
print(f"Test  {X_test.shape[0]:,} rows (churn {y_test.mean():.4f})")

Feature matrix : (7043, 19)
customerID in X: False
Churn in X     : False
Train 5,634 rows (churn 0.2654)
Test  1,409 rows (churn 0.2654)


In [14]:
candidates = train.build_candidates()
candidates["Logistic Regression"]

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. 

## 9. Model selection by cross-validation

The selection rule is fixed in `src/train.py` **before** any test-set result exists:
mean CV ROC-AUC first, then CV F1, then CV recall if the earlier gaps fall inside a 0.01
tolerance. Recall breaks the final tie because a missed churner costs more than an
unnecessary review.

In [15]:
cv_results = {
    name: train.cross_validate_model(pipeline, X_train, y_train)
    for name, pipeline in candidates.items()
}

pd.DataFrame(cv_results).T[
    ["cv_roc_auc_mean", "cv_roc_auc_std", "cv_f1_mean", "cv_recall_mean", "cv_precision_mean"]
].round(4)

,cv_roc_auc_mean,cv_roc_auc_std,cv_f1_mean,cv_recall_mean,cv_precision_mean
Logistic Regression,0.8460,0.0124,0.6286,0.8013,0.5174
Random Forest,0.8454,0.0091,0.6296,0.7632,0.5360


In [16]:
baseline_cv = train.cross_validate_model(train.build_baseline(), X_train, y_train)
print(f"Dummy baseline CV ROC-AUC: {baseline_cv['cv_roc_auc_mean']:.4f} (reference only)")

selected_model, justification = train.select_model(cv_results)
print("\nSELECTED:", selected_model)
print(justification)

Dummy baseline CV ROC-AUC: 0.5065 (reference only)

SELECTED: Logistic Regression
Cross-validated ROC-AUC (gap 0.0005) and F1 (gap 0.0010) were both within the 0.01 tolerance, so the rule fell through to mean CV recall. Logistic Regression was selected with 0.8013, because a missed churner costs more than an unnecessary review.


## 10. Held-out evaluation

The test set is scored **once**, after selection.

In [17]:
test_metrics = {}
for name, pipeline in candidates.items():
    pipeline.fit(X_train, y_train)
    proba = pipeline.predict_proba(X_test)[:, 1]
    predictions = (proba >= config.DECISION_THRESHOLD).astype(int)
    test_metrics[name] = evaluate.compute_metrics(y_test, predictions, proba)

pd.DataFrame(test_metrics).T.round(4)

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.7374,0.5034,0.7834,0.6130,0.8414
Random Forest,0.7608,0.5337,0.7834,0.6349,0.8417


In [18]:
from sklearn.metrics import confusion_matrix

selected_pipeline = candidates[selected_model]
proba = selected_pipeline.predict_proba(X_test)[:, 1]
predictions = (proba >= config.DECISION_THRESHOLD).astype(int)
matrix = confusion_matrix(y_test, predictions, labels=[0, 1])

pd.DataFrame(
    matrix,
    index=["actual: retained", "actual: churn"],
    columns=["predicted: retained", "predicted: churn"],
)

,predicted: retained,predicted: churn
actual: retained,746,289
actual: churn,81,293


**Reading the errors.**

- A **false negative** is a customer who churned but was never flagged. No retention
  review happens, so the opportunity is lost silently. This is the costly error.
- A **false positive** is a customer flagged for review who would have stayed. The cost is
  specialist time and the risk of an unnecessary contact, which is recoverable.

This asymmetry is why recall carries weight in selection and why accuracy alone would be
misleading on an imbalanced target.

## 11. Artifacts

`make train` writes the complete pipeline plus metadata, feature schema, reference rates
and model card into `deploy/artifacts/`. The application loads exactly that artifact.

In [19]:
import json

metadata = json.loads(config.METADATA_PATH.read_text(encoding="utf-8"))
print("Model      :", metadata["model_name"], "v" + metadata["model_version"])
print("Trained    :", metadata["training_timestamp_utc"])
print("Dataset SHA:", metadata["dataset_git_blob_sha"])
print("Metrics    :", json.dumps(metadata["metrics"], indent=2))

Model      : Logistic Regression v1.0.0
Trained    : 2026-07-23T14:05:31+00:00
Dataset SHA: 3de7a612d1609f25f21a455bda77948729369002
Metrics    : {
  "accuracy": 0.737402,
  "precision": 0.503436,
  "recall": 0.783422,
  "f1": 0.612971,
  "roc_auc": 0.841406
}


In [20]:
schema = schemas.load_feature_schema()
print("Predictors :", len(schema["features"]))
print("Excluded id:", schema["excluded_identifier"])
print("Order match:", schema["feature_order"] == config.FEATURE_COLUMNS)

Predictors : 19
Excluded id: customerID
Order match: True


## 12. Conclusions carried forward

1. The target is imbalanced, so recall, F1 and ROC-AUC — not accuracy — govern selection.
2. `TotalCharges` needs coercion and in-pipeline imputation; the raw file stays untouched.
3. `SeniorCitizen` is a binary category, and `customerID` is excluded from the features.
4. Contract term, internet service, tenure and payment method are the strongest observed
   signals in the sample.
5. Outputs are **decision support requiring human review**. No financial or
   customer-treatment decision may be automated from them, and no revenue or
   churn-reduction effect has been measured.